# Review 2 clean Transformer baseline on Kaggle T4

This notebook trains a **fresh** Transformer on a seeded, balanced subset of the combined flow collection. It uses the existing `src.data`, `src.models`, and `src.novelty` APIs. The input's per-row source-dataset provenance is unverified. This is development-scale clean training, not full-dataset training, continual learning, poisoning, or final research evidence.

The earlier one-epoch smoke run proved the wiring on a Tesla T4: 360/80/72 train/validation/test rows, validation accuracy 0.275, test accuracy 0.25, macro-F1 0.1915, and identical predictions after checkpoint reload. All 72 known-class test rows were flagged at confidence threshold 0.5. That result was preliminary because the model had only one epoch and nine test rows per broad class; the flags do not measure unknown-attack detection.

Import this `.ipynb` into Kaggle, enable **Internet: On** and **Accelerator: GPU**, and attach a private Kaggle Dataset containing `cic-collection.parquet` using **Add Input**. Run cells in order. The notebook writes a new timestamped folder under `/kaggle/working/`.


## 1. Check out the verified published source

This full SHA contains the existing data, model, novelty APIs, and the optional `fit(..., verbose=True)` progress reporting. A detached checkout keeps the run tied to the reviewed implementation even if the branch advances. Internet is required for cloning.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/DevChi1l/poison-resilient-continual-nids.git'
SOURCE_BRANCH = 'feature/unified-novelty'
SOURCE_SHA = '68135601a3ddd95832c57bb602f210f1a0e18a43'
REPO_DIR = Path('/kaggle/working/poison-resilient-continual-nids-clean')
if REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} already exists; inspect it or use a fresh Kaggle session before rerunning.')
subprocess.run(['git', 'clone', '--branch', SOURCE_BRANCH, '--single-branch', REPOSITORY_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=REPO_DIR, check=True)
checked_out_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert checked_out_sha == SOURCE_SHA
sys.path.insert(0, str(REPO_DIR))
print('Verified source SHA:', checked_out_sha)


## 2. Record Kaggle's software and require CUDA

Use Kaggle's supplied CUDA-enabled PyTorch. This cell checks imports, records actual package versions and GPU details, and stops if CUDA is unavailable. It does not install the repository's local requirements pins over Kaggle's working stack.


In [ ]:
from importlib.metadata import version
import platform
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib
import matplotlib.pyplot as plt
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this baseline. Enable a Kaggle GPU and restart the session.')
device = 'cuda'
environment = {
    'python': platform.python_version(),
    'packages': {name: version(name) for name in ('numpy', 'pandas', 'pyarrow', 'torch', 'matplotlib')},
    'torch_cuda_build': torch.version.cuda,
    'cuda_available': True,
    'gpu_name': torch.cuda.get_device_name(0),
}
assert np.asarray([1.0], dtype=np.float32).dtype == np.float32
assert pd.Series([1, 2]).sum() == 3
assert pa.array([1, 2]).to_pylist() == [1, 2]
assert torch.tensor([1.0], device=device).is_cuda
print('Environment:', environment)
print('Core package compatibility checks: passed')


## 3. Verify the new progress API on synthetic data

A tiny seeded fit checks `verbose=True` on Kaggle before scanning the real input. These generated numbers are only a functional check and are excluded from NIDS metrics. The later baseline constructs a separate, fresh model.


In [ ]:
from time import perf_counter
from src.data import prepare_sampled_dataset, sample_class_balanced_indices, stratified_split_indices
from src.models import ModelConfig, TabularTransformerClassifier
from src.novelty import detect_unknown

rng = np.random.default_rng(42)
synthetic_X = rng.normal(size=(16, 4)).astype(np.float32)
synthetic_y = np.asarray([10, 20] * 8, dtype=np.int64)
synthetic_config = ModelConfig(num_features=4, num_classes=2, hidden_dim=16, num_heads=2, num_layers=1, mlp_dim=32, batch_size=8, epochs=1, device=device, mixed_precision=False, seed=42, early_stopping_patience=None)
synthetic_model = TabularTransformerClassifier(synthetic_config, class_ids=[10, 20])
stage_start = perf_counter()
synthetic_history = synthetic_model.fit(synthetic_X[:12], synthetic_y[:12], synthetic_X[12:], synthetic_y[12:], verbose=True)
synthetic_seconds = perf_counter() - stage_start
assert synthetic_history.epochs_completed == 1
assert np.isfinite(synthetic_history.train_loss + synthetic_history.validation_loss).all()
print('Synthetic progress/API check passed in', round(synthetic_seconds, 3), 'seconds')


## 4. Locate and identify the attached Parquet file

Set `PARQUET_PATH` if the mounted input has a different path or several matching files are attached. The checksum from the completed smoke manifest is a reference for the same combined collection. This cell stops on a different checksum so we do not silently compare runs made from different files. Hashing reads the full file once.


In [ ]:
import hashlib

PARQUET_PATH = None  # Example: '/kaggle/input/my-private-dataset/cic-collection.parquet'
EXPECTED_INPUT_SHA256 = '666af53c788c79312b421c607289cb555c2335ae51ba39b2700a357595fa8dba'
candidates = sorted(Path('/kaggle/input').rglob('cic-collection.parquet'))
print('Matching attached files:', [str(path) for path in candidates])
if PARQUET_PATH is None:
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one attached cic-collection.parquet; found {len(candidates)}. Attach the file or set PARQUET_PATH explicitly.')
    parquet_path = candidates[0]
else:
    parquet_path = Path(PARQUET_PATH)
    if not parquet_path.is_file() or not parquet_path.is_relative_to(Path('/kaggle/input')):
        raise ValueError('PARQUET_PATH must name an attached file under /kaggle/input')
stage_start = perf_counter()
digest = hashlib.sha256()
with parquet_path.open('rb') as source_file:
    for chunk in iter(lambda: source_file.read(8 * 1024 * 1024), b''):
        digest.update(chunk)
input_sha256 = digest.hexdigest()
checksum_seconds = perf_counter() - stage_start
print('Attached input:', parquet_path, 'bytes:', parquet_path.stat().st_size, 'SHA-256:', input_sha256)
if input_sha256 != EXPECTED_INPUT_SHA256:
    raise RuntimeError('Input checksum differs from the reviewed smoke collection. Inspect the source before training.')


## 5. Sample 2,000 rows per broad class and fit train-only preprocessing

The completed smoke manifest observed at least 2,255 rows in each broad class, so 2,000 per class is feasible for that exact file. The existing seeded reservoir sampler verifies availability again during this run and scans the entire Parquet label column; bounded memory does not mean a short scan. The existing 70/15/15 splitter fixes membership with seed 42. Preprocessing fits imputation and scaling on training rows only, then applies them to validation and test. Test labels stay sealed until the evaluation stage.


In [ ]:
PER_CLASS_LIMIT = 2000
SAMPLE_SEED = 42
SPLIT_SEED = 42
stage_start = perf_counter()
sample = sample_class_balanced_indices(parquet_path, per_class_limit=PER_CLASS_LIMIT, seed=SAMPLE_SEED)
sampling_seconds = perf_counter() - stage_start
stage_start = perf_counter()
split = stratified_split_indices(sample.labels, seed=SPLIT_SEED)
splitting_seconds = perf_counter() - stage_start
stage_start = perf_counter()
dataset = prepare_sampled_dataset(parquet_path, sample, split)
preprocessing_seconds = perf_counter() - stage_start
partitions = {'train': dataset.train, 'validation': dataset.validation, 'test': dataset.test}
assert len(dataset.feature_names) == 54 and len(dataset.class_names) == 8
for name, partition in partitions.items():
    assert partition.features.dtype == np.float32 and partition.labels.dtype == np.int64
    assert partition.features.shape[1] == 54 and np.isfinite(partition.features).all()
    print(name, 'features:', partition.features.shape, 'labels:', partition.labels.shape, 'finite:', bool(np.isfinite(partition.features).all()))
print('Broad class ID mapping:', dataset.class_names)
print('Available class counts:', sample.available_counts)
print('Data stage seconds:', {'sampling': sampling_seconds, 'splitting': splitting_seconds, 'preprocessing': preprocessing_seconds})


## 6. Train a fresh clean baseline with validation-loss selection

The earlier one-epoch smoke checkpoint is never loaded. This cell constructs a new 64-dimensional, two-layer Transformer, trains for at most 20 epochs, and prints each epoch through the existing `fit(..., verbose=True)` API. Validation loss controls patience 5. `fit` restores the best validation-loss state before returning; the test partition is not inspected during training or model selection. Mixed precision is initially disabled.


In [ ]:
from datetime import datetime, timezone

OUTPUT_DIR = Path('/kaggle/working') / ('review2_clean_baseline_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
model_config = ModelConfig(num_features=len(dataset.feature_names), num_classes=len(dataset.class_names), hidden_dim=64, num_heads=4, num_layers=2, mlp_dim=128, batch_size=128, learning_rate=0.001, epochs=20, early_stopping_patience=5, device='cuda', mixed_precision=False, seed=42)
model = TabularTransformerClassifier(model_config, class_ids=tuple(sorted(dataset.class_names)))
torch.cuda.synchronize()
stage_start = perf_counter()
history = model.fit(dataset.train.features, dataset.train.labels, dataset.validation.features, dataset.validation.labels, verbose=True)
torch.cuda.synchronize()
training_seconds = perf_counter() - stage_start
history_dict = history.to_dict()
assert 1 <= history.epochs_completed <= 20
assert history.best_epoch is not None and 1 <= history.best_epoch <= history.epochs_completed
assert history.best_epoch == int(np.argmin(history.validation_loss)) + 1
assert np.isfinite(history.train_loss + history.validation_loss + history.validation_accuracy).all()
print('Epochs completed:', history.epochs_completed, 'best validation-loss epoch:', history.best_epoch)
print('Training seconds:', training_seconds)


## 7. Plot the observed training curves

The curves show train and validation loss plus validation accuracy for each completed epoch. The best validation-loss epoch is marked. These are diagnostics for fitting and model selection; they are not a test-set tuning signal.


In [ ]:
epochs = np.arange(1, history.epochs_completed + 1)
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, history.train_loss, marker='o', label='Train loss')
axes[0].plot(epochs, history.validation_loss, marker='o', label='Validation loss')
axes[0].axvline(history.best_epoch, color='gray', linestyle='--', label='Best validation epoch')
axes[0].set(xlabel='Epoch', ylabel='Cross-entropy loss', title='Training and validation loss')
axes[0].legend()
axes[1].plot(epochs, history.validation_accuracy, marker='o', color='tab:green')
axes[1].axvline(history.best_epoch, color='gray', linestyle='--')
axes[1].set(xlabel='Epoch', ylabel='Accuracy', ylim=(0, 1), title='Validation accuracy')
figure.tight_layout()
figure.savefig(OUTPUT_DIR / 'training_curves.png', dpi=160)
plt.show()


## 8. Open the test partition once, after model selection

Now the best validation-loss model is fixed. This cell computes test classification metrics, per-class recall/F1, and a confusion matrix labeled by broad class. It verifies probability columns against `model.class_ids`. Confidence threshold 0.5 remains unchanged from the smoke run; flag counts are diagnostics only because all broad classes were known during training. Do not tune this configuration against the test metrics.


In [ ]:
NOVELTY_THRESHOLD = 0.5
stage_start = perf_counter()
test_predictions = model.predict(dataset.test.features)
test_probabilities = model.predict_proba(dataset.test.features)
test_seconds = perf_counter() - stage_start
test_labels = dataset.test.labels
class_ids = np.asarray(model.class_ids, dtype=np.int64)
assert test_predictions.shape == test_labels.shape
assert test_probabilities.shape == (len(test_labels), len(class_ids))
assert np.isfinite(test_probabilities).all()
assert np.allclose(test_probabilities.sum(axis=1), 1.0, atol=1e-5)
assert np.array_equal(test_predictions, class_ids[test_probabilities.argmax(axis=1)])
unknown_mask = detect_unknown(test_probabilities, NOVELTY_THRESHOLD)
confusion = np.asarray([[np.sum((test_labels == truth) & (test_predictions == predicted)) for predicted in class_ids] for truth in class_ids], dtype=np.int64)
per_class = {}
for index, class_id in enumerate(class_ids):
    tp = int(confusion[index, index])
    predicted_count = int(confusion[:, index].sum())
    support = int(confusion[index, :].sum())
    precision = tp / predicted_count if predicted_count else 0.0
    recall = tp / support if support else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    per_class[str(int(class_id))] = {'name': dataset.class_names[int(class_id)], 'precision': precision, 'recall': recall, 'f1': f1, 'support': support}
metrics = {'accuracy': float(np.trace(confusion) / len(test_labels)), 'macro_f1': float(np.mean([item['f1'] for item in per_class.values()])), 'class_ids': class_ids.tolist(), 'confusion_matrix': confusion.tolist(), 'per_class': per_class}
novelty_counts = {'flagged_unknown': int(unknown_mask.sum()), 'not_flagged': int((~unknown_mask).sum())}
print('Test accuracy:', metrics['accuracy'], 'macro-F1:', metrics['macro_f1'])
for class_id, item in per_class.items():
    print(class_id, item['name'], 'recall:', round(item['recall'], 4), 'F1:', round(item['f1'], 4), 'support:', item['support'])
print('Confidence flag counts only:', novelty_counts, 'threshold:', NOVELTY_THRESHOLD)
labels = [f"{int(class_id)}: {dataset.class_names[int(class_id)]}" for class_id in class_ids]
figure, axis = plt.subplots(figsize=(11, 9))
image = axis.imshow(confusion, cmap='Blues')
axis.set_xticks(np.arange(len(labels)), labels=labels, rotation=45, ha='right')
axis.set_yticks(np.arange(len(labels)), labels=labels)
axis.set(xlabel='Predicted class', ylabel='True class', title='Clean baseline test confusion matrix')
for row in range(len(labels)):
    for column in range(len(labels)):
        axis.text(column, row, str(confusion[row, column]), ha='center', va='center', color='white' if confusion[row, column] > confusion.max() / 2 else 'black')
figure.colorbar(image, ax=axis, label='Rows')
figure.tight_layout()
figure.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=160)
plt.show()


## 9. Save the selected model and reproducibility record

The timestamped output folder receives the best validation-loss model state returned by `fit`, training history, test metrics, original selected/split row indices, and fitted preprocessing values. The manifest links the actual source SHA, dataset checksum, environment, configuration, timings, and artifact names. Download the folder from Kaggle; generated artifacts and the dataset do not belong in Git.


In [ ]:
from dataclasses import asdict
import json

save_start = perf_counter()
checkpoint_path = OUTPUT_DIR / 'transformer_best_validation.pt'
model.save(checkpoint_path)
np.savez_compressed(OUTPUT_DIR / 'row_indices.npz', selected_original=sample.row_indices, train_positions=split.train, validation_positions=split.validation, test_positions=split.test, train_original=dataset.train.row_indices, validation_original=dataset.validation.row_indices, test_original=dataset.test.row_indices)
preprocessing_state = {'feature_names': list(dataset.preprocessor.feature_names), 'imputation_values': dataset.preprocessor.imputation_values.tolist(), 'means': dataset.preprocessor.means.tolist(), 'scales': dataset.preprocessor.scales.tolist(), 'class_names': {str(key): value for key, value in dataset.class_names.items()}}
(OUTPUT_DIR / 'preprocessing.json').write_text(json.dumps(preprocessing_state, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'training_history.json').write_text(json.dumps(history_dict, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
saving_seconds = perf_counter() - save_start
manifest = {
    'purpose': 'Review-2 clean Transformer baseline on a balanced subset; all broad classes known to training',
    'source_sha': checked_out_sha, 'source_branch': SOURCE_BRANCH,
    'environment': environment,
    'input': {'path': str(parquet_path), 'bytes': parquet_path.stat().st_size, 'sha256': input_sha256, 'description': 'combined flow collection; per-row source provenance unverified'},
    'sampling': {'per_class_limit': PER_CLASS_LIMIT, 'seed': SAMPLE_SEED, 'class_order': list(sample.class_order), 'available_counts': sample.available_counts},
    'split': {'seed': SPLIT_SEED, 'fractions': {'train': 0.70, 'validation': 0.15, 'test': 0.15}, 'shapes': {name: list(partition.features.shape) for name, partition in partitions.items()}},
    'model_config': asdict(model.config), 'class_ids': list(model.class_ids),
    'model_selection': {'criterion': 'minimum validation loss', 'best_epoch': history.best_epoch, 'epochs_completed': history.epochs_completed, 'test_used_for_selection': False},
    'novelty_threshold': NOVELTY_THRESHOLD, 'novelty_flag_counts_only': novelty_counts,
    'training_history': history_dict, 'metrics': metrics,
    'timings_seconds': {'synthetic_check': synthetic_seconds, 'input_sha256': checksum_seconds, 'sampling': sampling_seconds, 'splitting': splitting_seconds, 'preprocessing': preprocessing_seconds, 'training': training_seconds, 'test_evaluation': test_seconds, 'saving': saving_seconds},
    'artifacts': {'checkpoint': checkpoint_path.name, 'row_indices': 'row_indices.npz', 'preprocessing': 'preprocessing.json', 'history': 'training_history.json', 'metrics': 'metrics.json', 'training_curves': 'training_curves.png', 'confusion_matrix': 'confusion_matrix.png'},
    'checkpoint_reload_verified': False,
}
manifest_path = OUTPUT_DIR / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Saved clean baseline artifacts:', OUTPUT_DIR)


## 10. Verify reload and collect the observed run

Reload the saved checkpoint through the repository API and require identical test predictions. The manifest is marked verified only after this succeeds. Send the final summary and output folder back for review before choosing another model configuration or the next implementation task.


In [ ]:
reload_start = perf_counter()
reloaded_model = TabularTransformerClassifier.load(checkpoint_path, device='cuda')
reloaded_predictions = reloaded_model.predict(dataset.test.features)
reload_seconds = perf_counter() - reload_start
assert reloaded_model.class_ids == model.class_ids
assert np.array_equal(reloaded_predictions, test_predictions), 'Checkpoint reload changed test predictions'
manifest['checkpoint_reload_verified'] = True
manifest['timings_seconds']['checkpoint_reload'] = reload_seconds
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Checkpoint reload predictions: identical')
print('Source SHA:', checked_out_sha, 'GPU:', environment['gpu_name'])
print('Input SHA-256:', input_sha256)
print('Split shapes:', manifest['split']['shapes'], 'class mapping:', dataset.class_names)
print('Best epoch:', history.best_epoch, 'epochs completed:', history.epochs_completed)
print('Test accuracy:', metrics['accuracy'], 'macro-F1:', metrics['macro_f1'])
print('Confidence flag counts only:', novelty_counts)
print('Timings seconds:', manifest['timings_seconds'])
print('Output folder:', OUTPUT_DIR)
print('Manifest:', manifest_path)
